# 05 - 全量召回率优先 vs 精度优先（time + sid）

本 Notebook 在**全量数据**上分别跑两套工程向流水线，**模型与日志独立保存**（与 `04_run_pipelines.ipynb` 的 F1 最大化实验互不覆盖）。

| 策略 | marker 前缀 | 实验 YAML | run_name（产物目录） | 阈值策略 |
|------|-------------|-----------|----------------------|----------|
| **召回率优先** | `full_recall` / `full_recall_sid` | `exp_full_recall_*.yaml` | `runs/exp_tianchi_full_recall_*` | `threshold_profile: recall_guard` |
| **精度优先** | `full_precision` / `full_precision_sid` | `exp_full_precision_*.yaml` | `runs/exp_tianchi_full_precision_*` | `threshold_profile: precision_guard` |

**共享 preprocess**：两套策略使用相同 data 配置（`drop_empty_positive_windows: true`），仅跑 **一次** preprocess（marker `full_eng__preprocess`）。

**日志**：`runs/pipeline-logs/{marker}__{step}__{timestamp}.log`  
**模型**：`runs/exp_tianchi_full_recall_ddp/`、`runs/exp_tianchi_full_precision_ddp/` 等（各自 `.latest_train_session` / `.latest_infer_session`）。

项目 KPI（F1 最大化）见 **`04_run_pipelines.ipynb`**。

**Notebook 提示**：修改 Python 源码后请先重跑 setup 单元，或 **Kernel → Restart**。

In [ ]:
import os
from pathlib import Path

from dram2cnn.utils.notebook_bootstrap import notebook_setup
from dram2cnn.utils.pipeline_helpers import pipeline_log_dir, pipeline_marker_dir

PROJECT_ROOT = notebook_setup(cwd=Path.cwd().resolve())
LOG_DIR = pipeline_log_dir(PROJECT_ROOT)
MARKER_DIR = pipeline_marker_dir(PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
from dram2cnn.utils.config import load_experiment_config
from dram2cnn.utils.pipeline_helpers import (
    clear_exp_markers,
    preflight_full_scale_index,
    preflight_split_labels,
    preflight_splits_vs_model,
    processed_dir_for_exp,
    require_preprocess_manifest,
    run_step,
    splits_dir_for_exp,
)
from dram2cnn.utils.deployment_kpi import (
    RECALL_PRECISION_KPI_PAIRS,
    collect_kpi_rows,
    merge_all_strategies_kpi,
    write_kpi_csv,
)

# 实验 YAML / marker 前缀（全 notebook 共用）
ENG_EXP = "full_eng"
ENG_PREPROCESS_CFG = "configs/experiments/exp_full_recall_ddp.yaml"
RECALL_TIME_EXP = "full_recall"
RECALL_TIME_CFG = "configs/experiments/exp_full_recall_ddp.yaml"
RECALL_SID_EXP = "full_recall_sid"
RECALL_SID_CFG = "configs/experiments/exp_full_recall_sid.yaml"
PREC_TIME_EXP = "full_precision"
PREC_TIME_CFG = "configs/experiments/exp_full_precision_ddp.yaml"
PREC_SID_EXP = "full_precision_sid"
PREC_SID_CFG = "configs/experiments/exp_full_precision_sid.yaml"

print("pipeline helpers loaded (run_step, preflight, deployment_kpi, 实验常量)")

---
## 共享：全量索引 + 工程向 preprocess

使用 `exp_full_recall_ddp.yaml` 的 data 段（与 precision 实验一致：`drop_empty_positive_windows: true`）。

> 若已用 **04（F1）** 跑过 preprocess（`drop_empty_positive_windows: false`），请先 `rm -rf data/processed/* data/splits/*` 并 `clear_exp_markers('full_eng')`。

In [ ]:
_cfg = preflight_full_scale_index(ENG_PREPROCESS_CFG, root=PROJECT_ROOT, context="05_eng_preflight")

run_step(ENG_EXP, "build_index", "bash scripts/build_index.sh full")
run_step(ENG_EXP, "preprocess", f"bash scripts/preprocess.sh {ENG_PREPROCESS_CFG}")
run_step(ENG_EXP, "inspect_data", f"bash scripts/inspect_data.sh {ENG_PREPROCESS_CFG}")

_pd = processed_dir_for_exp(ENG_PREPROCESS_CFG, PROJECT_ROOT)
_sd = splits_dir_for_exp(ENG_PREPROCESS_CFG, PROJECT_ROOT)
print(f"工程向 preprocess 完成 → {_pd.relative_to(PROJECT_ROOT)}/ + {_sd.relative_to(PROJECT_ROOT)}/")

---
## 召回率优先（宁报勿漏）

- **time**：`exp_full_recall_ddp.yaml` → `runs/exp_tianchi_full_recall_ddp/`
- **sid**：需 resplit → `runs/exp_tianchi_full_recall_sid/`
- 阈值：`recall_guard`（recall ≥ 0.88 且 precision ≥ 下限）

In [ ]:
_cfg = load_experiment_config(PROJECT_ROOT / RECALL_TIME_CFG)
print("threshold_profile:", _cfg["train"].get("threshold_profile"))
print("threshold_search:", _cfg["train"].get("threshold_search"))

preflight_splits_vs_model(RECALL_TIME_CFG)
preflight_split_labels(RECALL_TIME_CFG)

run_step(RECALL_TIME_EXP, "train", f"bash scripts/train_ddp.sh {RECALL_TIME_CFG}")
run_step(RECALL_TIME_EXP, "infer", f"bash scripts/infer.sh {RECALL_TIME_CFG}")
run_step(RECALL_TIME_EXP, "evaluate", f"bash scripts/evaluate.sh {RECALL_TIME_CFG}")

print("召回率优先 time 完成 → runs/exp_tianchi_full_recall_ddp/")

In [ ]:
require_preprocess_manifest(
    ENG_PREPROCESS_CFG,
    root=PROJECT_ROOT,
    hint_cmd=f"bash scripts/preprocess.sh {ENG_PREPROCESS_CFG}",
)
run_step(RECALL_SID_EXP, "resplit", f"bash scripts/resplit.sh {RECALL_SID_CFG}")
preflight_splits_vs_model(RECALL_SID_CFG)
preflight_split_labels(RECALL_SID_CFG)

run_step(RECALL_SID_EXP, "train", f"bash scripts/train_ddp.sh {RECALL_SID_CFG}")
run_step(RECALL_SID_EXP, "infer", f"bash scripts/infer.sh {RECALL_SID_CFG}")
run_step(RECALL_SID_EXP, "evaluate", f"bash scripts/evaluate.sh {RECALL_SID_CFG}")

print("召回率优先 sid 完成 → runs/exp_tianchi_full_recall_sid/")

---
## 精度优先（高精度告警）

- **time**：`exp_full_precision_ddp.yaml` → `runs/exp_tianchi_full_precision_ddp/`
- **sid**：`exp_full_precision_sid.yaml` → `runs/exp_tianchi_full_precision_sid/`
- 阈值：`precision_guard`（precision ≥ 0.65 下最大化 recall）

In [ ]:
_cfg = load_experiment_config(PROJECT_ROOT / PREC_TIME_CFG)
print("threshold_profile:", _cfg["train"].get("threshold_profile"))

preflight_splits_vs_model(PREC_TIME_CFG)
preflight_split_labels(PREC_TIME_CFG)

run_step(PREC_TIME_EXP, "train", f"bash scripts/train_ddp.sh {PREC_TIME_CFG}")
run_step(PREC_TIME_EXP, "infer", f"bash scripts/infer.sh {PREC_TIME_CFG}")
run_step(PREC_TIME_EXP, "evaluate", f"bash scripts/evaluate.sh {PREC_TIME_CFG}")

print("精度优先 time 完成 → runs/exp_tianchi_full_precision_ddp/")

In [ ]:
require_preprocess_manifest(
    ENG_PREPROCESS_CFG,
    root=PROJECT_ROOT,
    hint_cmd=f"bash scripts/preprocess.sh {ENG_PREPROCESS_CFG}",
)
run_step(PREC_SID_EXP, "resplit", f"bash scripts/resplit.sh {PREC_SID_CFG}")
preflight_splits_vs_model(PREC_SID_CFG)
preflight_split_labels(PREC_SID_CFG)

run_step(PREC_SID_EXP, "train", f"bash scripts/train_ddp.sh {PREC_SID_CFG}")
run_step(PREC_SID_EXP, "infer", f"bash scripts/infer.sh {PREC_SID_CFG}")
run_step(PREC_SID_EXP, "evaluate", f"bash scripts/evaluate.sh {PREC_SID_CFG}")

print("精度优先 sid 完成 → runs/exp_tianchi_full_precision_sid/")

In [ ]:
# 召回 vs 精度：四路 deployment KPI 汇总
rows = collect_kpi_rows(RECALL_PRECISION_KPI_PAIRS, project_root_path=PROJECT_ROOT)
if rows:
    import pandas as pd

    cmp_df = pd.DataFrame(rows)
    print(cmp_df.to_string(index=False))
    out = PROJECT_ROOT / "outputs" / "tables"
    write_kpi_csv(rows, out / "full_recall_vs_precision_deployment_kpi.csv")
    merged = merge_all_strategies_kpi(tables_dir=out)
    print("\n已写入 outputs/tables/full_recall_vs_precision_deployment_kpi.csv")
    if merged:
        print(f"已更新汇总表 {merged.name}")

In [ ]:
print("=== 本 Notebook 相关 marker ===")
for prefix in ("full_eng", "full_recall", "full_recall_sid", "full_precision", "full_precision_sid"):
    for p in sorted(MARKER_DIR.glob(f"{prefix}__*.done")):
        print(p.name, "=>", p.read_text(encoding="utf-8").strip())

In [ ]:
# 重置 marker（可选）
# clear_exp_markers("full_eng")
# clear_exp_markers("full_recall")
# clear_exp_markers("full_recall_sid")
# clear_exp_markers("full_precision")
# clear_exp_markers("full_precision_sid")
print("未删除 marker。")